# Imports

In [1]:
source("local-functions.R")

## Functions

In [2]:
# 2. Define the logistic regression separation test

check_separation <- function(data, initial_model, contrasts) {

  model <- tryCatch({

    stats::glm(
      formula=initial_model,
      data = data,
      family = stats::binomial(link = "logit"),
      contrasts = contrasts,
      method = detectseparation::detect_separation
    )

  }, error = function(e) {
    message("Separation test error: ", e$message)
    return(NULL)
  })

  if (is.null(model)) return(NA)

  # TRUE = separation detected
  # FALSE = no separation detected

  any(is.infinite(coef(model)))
}

In [3]:
# 3. Generate all possible combinations of filters

return_combinations_of_filters = function(filters){
    filter_names <- names(filters)
    combinations <- list(character(0))
    for (k in seq_along(filter_names)) {
    
      combinations <- c(
        combinations,
        combn(filter_names, k, simplify = FALSE)
      )
    }
    return (combinations)
    }

In [4]:
# 4. Test every combination

loop_thru_combinations = function(combinations, filters, df_original, contrasts){
    results <- list()
    keep_masks <- list()
    
    for (i in seq_along(combinations)) {    
      selected_filters <- combinations[[i]]
        
      # Start with all observations retained    
      keep <- rep(TRUE, nrow(df_original))
    
      # Apply the selected filters    
      for (f in selected_filters) {
          keep <- keep & filters[[f]](df_original)    
      }
    
      # Create the filtered dataset    
      df_test <- df_original[keep, , drop = FALSE]
    
      # Check separation    
      separation <- check_separation(df_test, initial_model, contrasts)
    
      # Store results    
      results[[i]] <- data.frame(
        filters = if (length(selected_filters) == 0) {
          "None"
        } else {
          paste(selected_filters, collapse = " + ")
        },    
        n_filters = length(selected_filters),    
        n_removed = sum(!keep),    
        n_remaining = sum(keep),    
        separation = separation    
      )
    
      keep_masks[[i]] <- keep
    
    }
    
    results_df <- dplyr::bind_rows(results)
    
    # Keep combinations where separation has disappeared
    
    successful <- results_df |>
      dplyr::filter(!is.na(separation), separation == FALSE) |>
      dplyr::arrange(n_removed, n_filters)
    
    # Display the best solutions
    return (successful)
}

# Study 1

## What the non-participants said

In [5]:
df1 = haven::read_sav("../data/Pilot1.sav") |> 
    add_participation() |> 
    dplyr::filter(participation %in% c("Didn't participate","Didn't answer the question"))

# Source column names
variables <- c(paste0("G08Q43_SQ", sprintf("%03d", 1:11)), "G08Q43_other")

# Descriptive labels
labels <- c(
  "Concern about the possible discomfort associated with the procedures",
  "Concern about possible health risk associated with procedures",
  "Having to travel too far to take part in procedures",
  "Financial reasons",
  "Time constraints",
  "Unwillingness to have my H. pylori infection treated",
  "Belief in my good health and the unnecessity of these procedures",
  "Mistrust towards the healthcare system/ my healthcare provider",
  "Do not understand the methodology of the study",
  "Do not understand the purpose of the study",
  "Not seeing a benefit to participation",
  "Other"
)

names(labels) <- variables

# Convert any non-empty value in G04Q16_other to "Y"
df1$G08Q43_other[
  !is.na(df1$G08Q43_other) &
    trimws(df1$G08Q43_other) != ""
] <- "Y"

# Extract relevant columns
response_data <- df1[, variables]

# Convert Y to 1 and empty/missing values to 0
response_matrix <- as.matrix(
  as.data.frame(
    lapply(response_data, function(x) {
      as.integer(!is.na(x) & trimws(as.character(x)) == "Y")
    })
  )
)

# Assign descriptive labels
colnames(response_matrix) <- labels

# Calculate pairwise counts
cooccurrence_matrix <- crossprod(response_matrix)

# Get ordering based on diagonal counts (descending)
ordering <- order(
  diag(cooccurrence_matrix),
  decreasing = TRUE
)

# Reorder rows and columns identically
cooccurrence_matrix <- cooccurrence_matrix[
  ordering,
  ordering,
  drop = FALSE
]

# Convert to dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

# Convert to a dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

IRdisplay::display_markdown(
  "### Why wouldn’t you accept to participate in the study?"
)

html_table <- knitr::kable(
  cooccurrence_table,
  format = "html",
  align = "c",
  caption = "Number of respondents selecting each pair of reasons"
)

IRdisplay::display_html(
  as.character(html_table)
)

### Why wouldn’t you accept to participate in the study?

,Time constraints,Other,Concern about the possible discomfort associated with the procedures,Belief in my good health and the unnecessity of these procedures,Concern about possible health risk associated with procedures,Not seeing a benefit to participation,Having to travel too far to take part in procedures,Mistrust towards the healthcare system/ my healthcare provider,Do not understand the purpose of the study,Financial reasons,Unwillingness to have my H. pylori infection treated,Do not understand the methodology of the study
Time constraints,19,0,0,2,0,1,1,0,0,0,0,0
Other,0,8,0,0,0,0,0,0,0,0,0,0
Concern about the possible discomfort associated with the procedures,0,0,6,0,1,0,0,0,0,0,0,0
Belief in my good health and the unnecessity of these procedures,2,0,0,3,0,0,0,1,1,0,0,0
Concern about possible health risk associated with procedures,0,0,1,0,2,0,0,0,0,0,0,0
Not seeing a benefit to participation,1,0,0,0,0,2,0,0,0,0,0,0
Having to travel too far to take part in procedures,1,0,0,0,0,0,1,0,0,0,0,0
Mistrust towards the healthcare system/ my healthcare provider,0,0,0,1,0,0,0,1,1,0,0,0
Do not understand the purpose of the study,0,0,0,1,0,0,0,1,1,0,0,0
Financial reasons,0,0,0,0,0,0,0,0,0,0,0,0


## Data processing

In [6]:
df1 = haven::read_sav("../data/Pilot1.sav")
nrow(df1)

[1] 716

In [7]:
df1_initial = haven::read_sav("../data/Pilot1.sav") |> 
    add_participation() |>
    add_gender() |> 
    add_education() |> 
    add_health() |> 
    add_family_history(1) |> 
    add_knowing() |> 
    add_prior_pilory() |>
    add_awareness_pilory() |> 
    add_awareness_GC(1)

variables1 <- c(
  "gender", "education", "health", "family_history",
  "knowing", "prior_pilory", "awareness_pilory", "awareness_GC"
)

df1_initial <- df1_initial |>
  dplyr::mutate(
    dplyr::across(
      dplyr::all_of(variables1),
      ~ forcats::fct_collapse(
        .x,
        "Didn't answer" = c(
          "I prefer not to answer",
          "Didn't answer the question",
          "I prefer not to respond"
        )
      )
    )
  )

Warning message:
“There were 8 warnings in `dplyr::mutate()`.
The first warning was:
ℹ In argument: `dplyr::across(...)`.
Caused by warning:
! Unknown levels in `f`: I prefer not to respond
ℹ Run `dplyr::last_dplyr_warnings()` to see the 7 remaining warnings.”


## Multivariate model

In [29]:
initial_model <- as.formula(
  participation ~ gender + education + health + family_history + knowing + prior_pilory + awareness_pilory + awareness_GC
)

### Ensuring model feasibility

In [9]:
df_original <- df1_initial

# 1. Define the eight filters

filters <- list(

  gender = function(d) {
    d$gender %in% c("Male", "Female")
  },

  education = function(d) {d$education != "Didn't answer"},
  health = function(d) {!(d$health %in% c("Bad", "I don't know", "Didn't answer"))},
  family_history = function(d) {d$family_history != "Didn't answer"},
  knowing = function(d) {d$knowing != "Didn't answer"},
  prior_pilory = function(d) {d$prior_pilory %in% c("No", "Other", "Didn't answer")},
  awareness_pilory = function(d) {d$awareness_pilory != "Didn't answer"},
  awareness_GC = function(d) {d$awareness_GC != "Didn't answer"}

)

# 2. generate combinations
combinations = return_combinations_of_filters(filters)

In [10]:
contrasts = list(
        gender = "contr.treatment",
        education = "contr.treatment",
        health = "contr.poly",
        family_history = "contr.treatment",
        knowing = "contr.treatment",
        prior_pilory = "contr.treatment",
        awareness_pilory = "contr.treatment",
        awareness_GC = "contr.treatment"
      )
successful = loop_thru_combinations(combinations, filters, df_original, contrasts)
successful

filters,n_filters,n_removed,n_remaining,separation
<chr>,<int>,<int>,<int>,<lgl>
gender + health + family_history + knowing + prior_pilory + awareness_pilory,6,24,692,FALSE
gender + health + family_history + knowing + prior_pilory + awareness_pilory + awareness_GC,7,24,692,FALSE
gender + education + health + family_history + prior_pilory + awareness_pilory,6,28,688,FALSE
gender + education + health + family_history + knowing + prior_pilory + awareness_pilory,7,28,688,FALSE
gender + education + health + family_history + prior_pilory + awareness_pilory + awareness_GC,7,28,688,FALSE
gender + education + health + family_history + knowing + prior_pilory + awareness_pilory + awareness_GC,8,28,688,FALSE


In [11]:
# confirm separation
df1_tmp = df1_initial |> dplyr::filter(gender %in% c("Male", "Female"),
                                      #education != "Didn't answer",
                                      !(health %in% c("Bad", "I don't know", "Didn't answer")),
                                      family_history != "Didn't answer",
                                      knowing != "Didn't answer",
                                      prior_pilory %in% c("No", "Other", "Didn't answer"),
                                      awareness_pilory != "Didn't answer",
                                      #awareness_GC != "Didn't answer"
                                     )
nrow(df1_tmp)

model_separation <- stats::glm(
    formula=initial_model,
  data = df1_tmp,
  family = stats::binomial(link = "logit"),
  contrasts = contrasts,
  method = detectseparation::detect_separation
)

model_separation

[1] 692

Implementation: ROI | Solver: lpsolve 
Separation: FALSE 
Existence of maximum likelihood estimates
                                (Intercept) 
                                          0 
                                 genderMale 
                                          0 
         educationHigh school or vocational 
                                          0 
educationPrimary/did not finish high school 
                                          0 
                     educationDidn't answer 
                                          0 
                                   health.L 
                                          0 
                                   health.Q 
                                          0 
                           family_historyNo 
                                          0 
                 family_historyI don't know 
                                          0 
                                 knowingYes 
                                          0 


### Logistic Model

In [30]:
# Fit logistic regression
model_participation <- stats::glm(
  formula=stats::update(initial_model, . ~ . -awareness_GC),
  data = df1_tmp,
  family = stats::binomial(link = "logit"),
  contrasts = contrasts
  )

# Model summary
summary(model_participation)

Warning message in model.matrix.default(mt, mf, contrasts):
“variable 'age' is absent, its contrast will be ignored”
Warning message in model.matrix.default(mt, mf, contrasts):
“variable 'awareness_GC' is absent, its contrast will be ignored”
Warning message in model.matrix.default(mt, mf, contrasts):
“variable 'prior_endoscopy' is absent, its contrast will be ignored”
Warning message in model.matrix.default(mt, mf, contrasts):
“variable 'prior_GC_testing' is absent, its contrast will be ignored”



Call:
stats::glm(formula = stats::update(initial_model, . ~ . - awareness_GC), 
    family = stats::binomial(link = "logit"), data = df1_tmp, 
    contrasts = contrasts)

Coefficients:
                                            Estimate Std. Error z value
(Intercept)                                 -4.38393    0.58263  -7.524
genderMale                                   0.19543    0.36076   0.542
educationHigh school or vocational           1.90294    0.37164   5.120
educationPrimary/did not finish high school  1.91849    0.94059   2.040
educationDidn't answer                       2.84897    1.39482   2.043
health.L                                    -0.59595    0.49999  -1.192
health.Q                                    -0.25972    0.32848  -0.791
family_historyNo                             0.60529    0.36452   1.661
family_historyI don't know                   1.22421    0.60255   2.032
knowingYes                                  -0.30257    0.39735  -0.761
knowingI don’t know   

In [31]:
results_participation <- broom::tidy(
  model_participation,
  exponentiate = TRUE,
  conf.int = TRUE
) |>
  dplyr::select(
    term,
    estimate,
    conf.low,
    conf.high,
    p.value
  ) |>
  dplyr::rename(
    Variable = term,
    OR = estimate,
    CI_lower = conf.low,
    CI_upper = conf.high,
    p_value = p.value
  )

results_participation

Variable,OR,CI_lower,CI_upper,p_value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),0.01247623,0.003711767,0.03665663,5.298788e-14
genderMale,1.21583347,0.593726564,2.46278880,5.880115e-01
educationHigh school or vocational,6.70559000,3.232344638,13.98826134,3.048726e-07
educationPrimary/did not finish high school,6.81069432,0.845665504,39.80775943,4.138246e-02
educationDidn't answer,17.26990541,0.657948288,242.50240177,4.109885e-02
health.L,0.55104143,0.189796051,1.38970853,2.332924e-01
health.Q,0.77127064,0.390714703,1.43767822,4.291456e-01
family_historyNo,1.83178655,0.899543317,3.79318923,9.680730e-02
family_historyI don't know,3.40148375,0.980362332,10.69482059,4.218333e-02


### Table of results

In [32]:
summary(df1_tmp$participation)

Participated         Didn't participate 
                       648                         44 
Didn't answer the question 
                         0

In [33]:
gt_table = results_participation |>
  gt::gt() |>
  gt::tab_header(
    title = "Study 1. Effects estimates for participation",
    subtitle = "Based on 648 participants and 44 non-participants"
  ) |>
  gt::cols_label(
    Variable = "Variable",
    OR = "Odds ratio",
    CI_lower = "95% CI (lower)",
    CI_upper = "95% CI (upper)",
    p_value = "p-value"
  ) |>
  gt::fmt_number(
    columns = c(OR, CI_lower, CI_upper),
    decimals = 2
  ) |>
  gt::fmt_number(
    columns = p_value,
    decimals = 3
  ) |>
  gt::tab_options(
    table.font.size = gt::px(13)
  )

# htmltools::HTML(gt::as_raw_html(gt_table))
IRdisplay::display_html(gt::as_raw_html(gt_table))

<table class="gt_table" data-quarto-disable-processing="false" data-quarto-bootstrap="false" style="-webkit-font-smoothing: antialiased; -moz-osx-font-smoothing: grayscale; font-family: system-ui, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif, 'Apple Color Emoji', 'Segoe UI Emoji', 'Segoe UI Symbol', 'Noto Color Emoji'; display: table; border-collapse: collapse; line-height: normal; margin-left: auto; margin-right: auto; color: #333333; font-size: 13px; font-weight: normal; font-style: normal; background-color: #FFFFFF; width: auto; border-top-style: solid; border-top-width: 2px; border-top-color: #A8A8A8; border-right-style: none; border-right-width: 2px; border-right-color: #D3D3D3; border-bottom-style: solid; border-bottom-width: 2px; border-bottom-color: #A8A8A8; border-left-style: none; border-left-width: 2px; border-left-color: #D3D3D3;" bgcolor="#FFFFFF">
 
 
 Study 1. Effects estimates for participation 
 
 
 <td colspan="5" class="gt_heading gt_subtitle gt_font_normal gt_bottom_border" style="border-style: none; color: #333333; font-size: 85%; padding-top: 3px; padding-bottom: 5px; padding-left: 5px; padding-right: 5px; border-top-color: #FFFFFF; border-top-width: 0; background-color: #FFFFFF; text-align: center; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; border-bottom-style: solid; border-bottom-width: 2px; border-bottom-color: #D3D3D3; font-weight: normal;" bgcolor="#FFFFFF" align="center">Based on 648 participants and 44 non-participants 
 
 
 <th class="gt_col_heading gt_columns_bottom_border gt_left" rowspan="1" colspan="1" scope="col" id="Variable" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: left;" bgcolor="#FFFFFF" valign="bottom" align="left">Variable 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="OR" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: right; font-variant-numeric: tabular-nums;" bgcolor="#FFFFFF" valign="bottom" align="right">Odds ratio 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="CI_lower" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: right; font-variant-numeric: tabular-nums;" bgcolor="#FFFFFF" valign="bottom" align="right">95% CI (lower) 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="CI_upper" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; tex

# Study 2

## What the non-participants said

In [15]:
df2 = haven::read_sav("../data/Pilot2.sav") |> 
    add_participation() |> 
    dplyr::filter(participation %in% c("Didn't participate","Didn't answer the question"))

# Source column names
variables <- c(paste0("G06Q24_SQ", sprintf("%03d", 1:6)), "G06Q24_other")

# Descriptive labels
labels <- c(
  "Concern about the possible discomfort associated with the procedures",
  "Concern about possible health risks",
  "Having to travel too far to take part in procedures",
  "Financial reasons",
  "Belief in my good health and the unnecessity of these procedures",
  "Mistrust towards the healthcare system/ my healthcare provider",
  "Other"
)

names(labels) <- variables

# Convert any non-empty value in G04Q16_other to "Y"
df2$G06Q24_other[
  !is.na(df2$G06Q24_other) &
    trimws(df2$G06Q24_other) != ""
] <- "Y"

# Extract relevant columns
response_data <- df2[, variables]

# Convert Y to 1 and empty/missing values to 0
response_matrix <- as.matrix(
  as.data.frame(
    lapply(response_data, function(x) {
      as.integer(!is.na(x) & trimws(as.character(x)) == "Y")
    })
  )
)

# Assign descriptive labels
colnames(response_matrix) <- labels

# Calculate pairwise counts
cooccurrence_matrix <- crossprod(response_matrix)

# Get ordering based on diagonal counts (descending)
ordering <- order(
  diag(cooccurrence_matrix),
  decreasing = TRUE
)

# Reorder rows and columns identically
cooccurrence_matrix <- cooccurrence_matrix[
  ordering,
  ordering,
  drop = FALSE
]

# Convert to dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

# Convert to a dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

IRdisplay::display_markdown(
  "### Why wouldn’t you accept to participate in upper endoscopy and colonoscopy procedures?"
)

html_table <- knitr::kable(
  cooccurrence_table,
  format = "html",
  align = "c",
  caption = "Number of respondents selecting each pair of reasons"
)

IRdisplay::display_html(
  as.character(html_table)
)

### Why wouldn’t you accept to participate in upper endoscopy and colonoscopy procedures?

,Concern about the possible discomfort associated with the procedures,Other,Concern about possible health risks,Belief in my good health and the unnecessity of these procedures,Having to travel too far to take part in procedures,Mistrust towards the healthcare system/ my healthcare provider,Financial reasons
Concern about the possible discomfort associated with the procedures,69,18,25,16,2,2,0
Other,18,41,12,9,1,6,0
Concern about possible health risks,25,12,39,0,2,3,0
Belief in my good health and the unnecessity of these procedures,16,9,0,39,0,1,0
Having to travel too far to take part in procedures,2,1,2,0,10,0,0
Mistrust towards the healthcare system/ my healthcare provider,2,6,3,1,0,9,0
Financial reasons,0,0,0,0,0,0,0


In [16]:
df2 = haven::read_sav("../data/Pilot2.sav") |> 
    add_participation() |> 
    dplyr::filter(participation %in% c("Didn't participate","Didn't answer the question"))

# Source column names
variables <- c(paste0("G06Q25_SQ", sprintf("%03d", 1:4)), "G06Q25_other")

# Descriptive labels
labels <- c(
  "I am more concerned about my risk of developing colorectal cancer",
  "I find the colonoscopy procedure safer and less invasive than upper endoscopy",
  "Because upper endoscopy tests for stomach cancer and there is no standard screening programme for stomach cancer",
  "I would prefer to undergo these procedures separately",
  "Other"
)

names(labels) <- variables

# Convert any non-empty value in G04Q16_other to "Y"
df2$G06Q25_other[
  !is.na(df2$G06Q25_other) &
    trimws(df2$G06Q25_other) != ""
] <- "Y"

# Extract relevant columns
response_data <- df2[, variables]

# Convert Y to 1 and empty/missing values to 0
response_matrix <- as.matrix(
  as.data.frame(
    lapply(response_data, function(x) {
      as.integer(!is.na(x) & trimws(as.character(x)) == "Y")
    })
  )
)

# Assign descriptive labels
colnames(response_matrix) <- labels

# Calculate pairwise counts
cooccurrence_matrix <- crossprod(response_matrix)

# Get ordering based on diagonal counts (descending)
ordering <- order(
  diag(cooccurrence_matrix),
  decreasing = TRUE
)

# Reorder rows and columns identically
cooccurrence_matrix <- cooccurrence_matrix[
  ordering,
  ordering,
  drop = FALSE
]

# Convert to dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

# Convert to a dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

IRdisplay::display_markdown(
  "### Why wouldn’t you accept to participate in the study?"
)

html_table <- knitr::kable(
  cooccurrence_table,
  format = "html",
  align = "c",
  caption = "Number of respondents selecting each pair of reasons"
)

IRdisplay::display_html(
  as.character(html_table)
)

### Why wouldn’t you accept to participate in the study?

,I am more concerned about my risk of developing colorectal cancer,I would prefer to undergo these procedures separately,Other,I find the colonoscopy procedure safer and less invasive than upper endoscopy,Because upper endoscopy tests for stomach cancer and there is no standard screening programme for stomach cancer
I am more concerned about my risk of developing colorectal cancer,36,5,1,2,1
I would prefer to undergo these procedures separately,5,31,1,0,0
Other,1,1,25,0,0
I find the colonoscopy procedure safer and less invasive than upper endoscopy,2,0,0,9,1
Because upper endoscopy tests for stomach cancer and there is no standard screening programme for stomach cancer,1,0,0,1,5


## Data processing

In [17]:
df2 = haven::read_sav("../data/Pilot2.sav")
nrow(df2)

[1] 1254

In [18]:
df2_initial = haven::read_sav("../data/Pilot2.sav") |> 
    add_participation() |> 
    dplyr::mutate(
        participation = forcats::fct_collapse(participation,"Didn't participate" = c("Didn't participate", "Didn't answer the question"))) |>
    add_age() |>
    add_gender() |>
    add_education() |>
    add_health() |>
    add_family_history(2) |> 
    add_awareness_GC(2) |> 
    add_prior_endoscopy() |>
    add_prior_GC_testing() |>
    dplyr::mutate(
        health = forcats::fct_collapse(health,"Bad" = c("Bad", "Very bad")),
        awareness_GC = forcats::fct_collapse(awareness_GC,"No" = c("No", "I don’t know")))

variables1 <- c(
  "age", "gender", "education", "health", "family_history",
   "awareness_GC", "prior_endoscopy", "prior_GC_testing"
)

df2_initial <- df2_initial |>
  dplyr::mutate(
    dplyr::across(
      dplyr::all_of(variables1),
      ~ forcats::fct_collapse(
        .x,
        "Didn't answer" = c(
          "I prefer not to answer",
          "Didn't answer the question",
          "I prefer not to respond"
        )
      )
    )
  )


Warning message:
“There were 8 warnings in `dplyr::mutate()`.
The first warning was:
ℹ In argument: `dplyr::across(...)`.
Caused by warning:
! Unknown levels in `f`: I prefer not to respond
ℹ Run `dplyr::last_dplyr_warnings()` to see the 7 remaining warnings.”


## Multivariate model

In [19]:
initial_model <- as.formula(
  participation ~ age + gender + education + health + family_history + awareness_GC + prior_endoscopy + prior_GC_testing
)

### Ensuring model feasibility

In [20]:
df_original <- df2_initial

# 1. Define the eight filters

filters <- list(

  age = function(d) {
    !(d$age %in% c(
      "I prefer not to answer",
      "Didn't answer the question"
    ))
  },

  gender = function(d) {
    d$gender %in% c("Male", "Female")
  },

  education = function(d) {
    !(d$education %in% c(
      "I prefer not to answer",
      "Didn't answer the question"
    ))
  },

  health = function(d) {
    !(d$health %in% c(
      "I prefer not to respond",
      "Didn't answer the question"
    ))
  },

  family_history = function(d) {
    !(d$family_history %in% c(
      "I prefer not to answer",
      "Didn't answer the question"
    ))
  },

  awareness_GC = function(d) {
    !(d$awareness_GC %in% c(
      "I prefer not to answer",
      "Didn't answer the question"
    ))
  },

  prior_endoscopy = function(d) {
    !(d$prior_endoscopy %in% c(
      "I prefer not to answer",
      "Didn't answer the question"
    ))
  },

  prior_GC_testing = function(d) {
    !(d$prior_GC_testing %in% c(
      "I prefer not to answer",
      "Didn't answer the question"
    ))
  }
)

# 2. generate combinations
combinations = return_combinations_of_filters(filters)

In [21]:
  contrasts = list(
    age = "contr.poly",
    gender = "contr.treatment",
    education = "contr.treatment",
    health = "contr.poly",
    family_history = "contr.treatment",
    awareness_GC = "contr.treatment",
    prior_endoscopy = "contr.treatment",
    prior_GC_testing = "contr.treatment"
  )
successful = loop_thru_combinations(combinations, filters, df_original, contrasts)
successful

filters,n_filters,n_removed,n_remaining,separation
<chr>,<int>,<int>,<int>,<lgl>
None,0,0,1254,FALSE
age,1,0,1254,FALSE
education,1,0,1254,FALSE
health,1,0,1254,FALSE
family_history,1,0,1254,FALSE
awareness_GC,1,0,1254,FALSE
prior_endoscopy,1,0,1254,FALSE
prior_GC_testing,1,0,1254,FALSE
age + education,2,0,1254,FALSE


In [22]:
# confirm separation
#df2_tmp = df2_initial |> dplyr::filter(gender %in% c("Male", "Female"),
#                                      #education != "Didn't answer",
#                                      !(health %in% c("Bad", "I don't know", "Didn't answer")),
#                                      family_history != "Didn't answer",
#                                      knowing != "Didn't answer",
#                                      prior_pilory %in% c("No", "Other", "Didn't answer"),
#                                      awareness_pilory != "Didn't answer",
#                                      #awareness_GC != "Didn't answer"
#                                     )
nrow(df_original)

model_separation <- stats::glm(
    formula=initial_model,
  data = df_original,
  family = stats::binomial(link = "logit"),
  contrasts = contrasts,
  method = detectseparation::detect_separation
)

model_separation

[1] 1254

Implementation: ROI | Solver: lpsolve 
Separation: FALSE 
Existence of maximum likelihood estimates
                                (Intercept) 
                                          0 
                                      age.L 
                                          0 
                                      age.Q 
                                          0 
                                      age.C 
                                          0 
                                      age^4 
                                          0 
                                      age^5 
                                          0 
                                 genderMale 
                                          0 
                        genderDidn't answer 
                                          0 
         educationHigh school or vocational 
                                          0 
educationPrimary/did not finish high school 
                                          0 


### Logistic Model

In [23]:
# Fit logistic regression
model_participation <- stats::glm(
    formula=initial_model,
  data = df_original,
  family = stats::binomial(link = "logit"),
  contrasts = contrasts
)

# Model summary
summary(model_participation)


Call:
stats::glm(formula = initial_model, family = stats::binomial(link = "logit"), 
    data = df_original, contrasts = contrasts)

Coefficients:
                                            Estimate Std. Error z value
(Intercept)                                 -3.25230    0.44529  -7.304
age.L                                       -0.44593    0.95972  -0.465
age.Q                                       -0.66788    0.87239  -0.766
age.C                                       -0.51994    0.62316  -0.834
age^4                                       -0.20472    0.37421  -0.547
age^5                                       -0.12852    0.24288  -0.529
genderMale                                  -0.25779    0.21473  -1.201
genderDidn't answer                         -1.10872    1.63493  -0.678
educationHigh school or vocational           0.44658    0.25392   1.759
educationPrimary/did not finish high school -0.49788    0.41888  -1.189
educationDidn't answer                       0.29229    0.53

In [24]:
results_participation <- broom::tidy(
  model_participation,
  exponentiate = TRUE,
  conf.int = TRUE
) |>
  dplyr::select(
    term,
    estimate,
    conf.low,
    conf.high,
    p.value
  ) |>
  dplyr::rename(
    Variable = term,
    OR = estimate,
    CI_lower = conf.low,
    CI_upper = conf.high,
    p_value = p.value
  )

results_participation

Variable,OR,CI_lower,CI_upper,p_value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),0.03868517,0.01536442,0.08717256,2.797689e-13
age.L,0.64023172,0.08420062,2.96077344,6.421874e-01
age.Q,0.51279676,0.08151098,2.07583738,4.439347e-01
age.C,0.59455355,0.16073137,1.66898209,4.040710e-01
age^4,0.81487185,0.37806942,1.59792641,5.843254e-01
age^5,0.87939839,0.54196517,1.40714209,5.967045e-01
genderMale,0.77275903,0.50574333,1.17558599,2.299374e-01
genderDidn't answer,0.32998263,0.01165331,5.43007561,4.976807e-01
educationHigh school or vocational,1.56295317,0.96082821,2.60839626,7.861813e-02


### Table of results

In [25]:
summary(df_original$participation)

Participated Didn't participate 
              1079                175

In [26]:
gt_table = results_participation |>
  gt::gt() |>
  gt::tab_header(
    title = "Study 2. Effects estimates for participation",
    subtitle = "Based on 1079 participants and 175 non-participants"
  ) |>
  gt::cols_label(
    Variable = "Variable",
    OR = "Odds ratio",
    CI_lower = "95% CI (lower)",
    CI_upper = "95% CI (upper)",
    p_value = "p-value"
  ) |>
  gt::fmt_number(
    columns = c(OR, CI_lower, CI_upper),
    decimals = 2
  ) |>
  gt::fmt_number(
    columns = p_value,
    decimals = 3
  ) |>
  gt::tab_options(
    table.font.size = gt::px(13)
  )

# htmltools::HTML(gt::as_raw_html(gt_table))
IRdisplay::display_html(gt::as_raw_html(gt_table))

<table class="gt_table" data-quarto-disable-processing="false" data-quarto-bootstrap="false" style="-webkit-font-smoothing: antialiased; -moz-osx-font-smoothing: grayscale; font-family: system-ui, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif, 'Apple Color Emoji', 'Segoe UI Emoji', 'Segoe UI Symbol', 'Noto Color Emoji'; display: table; border-collapse: collapse; line-height: normal; margin-left: auto; margin-right: auto; color: #333333; font-size: 13px; font-weight: normal; font-style: normal; background-color: #FFFFFF; width: auto; border-top-style: solid; border-top-width: 2px; border-top-color: #A8A8A8; border-right-style: none; border-right-width: 2px; border-right-color: #D3D3D3; border-bottom-style: solid; border-bottom-width: 2px; border-bottom-color: #A8A8A8; border-left-style: none; border-left-width: 2px; border-left-color: #D3D3D3;" bgcolor="#FFFFFF">
 
 
 Study 2. Effects estimates for participation 
 
 
 <td colspan="5" class="gt_heading gt_subtitle gt_font_normal gt_bottom_border" style="border-style: none; color: #333333; font-size: 85%; padding-top: 3px; padding-bottom: 5px; padding-left: 5px; padding-right: 5px; border-top-color: #FFFFFF; border-top-width: 0; background-color: #FFFFFF; text-align: center; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; border-bottom-style: solid; border-bottom-width: 2px; border-bottom-color: #D3D3D3; font-weight: normal;" bgcolor="#FFFFFF" align="center">Based on 1079 participants and 175 non-participants 
 
 
 <th class="gt_col_heading gt_columns_bottom_border gt_left" rowspan="1" colspan="1" scope="col" id="Variable" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: left;" bgcolor="#FFFFFF" valign="bottom" align="left">Variable 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="OR" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: right; font-variant-numeric: tabular-nums;" bgcolor="#FFFFFF" valign="bottom" align="right">Odds ratio 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="CI_lower" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: right; font-variant-numeric: tabular-nums;" bgcolor="#FFFFFF" valign="bottom" align="right">95% CI (lower) 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="CI_upper" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; t